# SA3 LoRA Training with underfit

**Your setup, saved.** Run the cells top to bottom. Each one says what it does and what
success looks like.

| | |
|---|---|
| Google account | `w2@2w12.one` (Colab Pro + Drive + HuggingFace must all match) |
| Model | `sa3-medium` |
| Dataset | Dune OST — 38 tracks, captioned by mira |
| Trigger word | `zvq` |

### Before you start

1. **Set the GPU.** Runtime → Change runtime type → **L4 GPU** → Save.
2. **Put your dataset zip on Drive** (once): drag `dune-ost-latents-same-l.zip` into
   *My Drive → Colab Notebooks*.

> ⚠️ **When you're done, Runtime → Disconnect and delete runtime.** Colab keeps billing
> compute units while the VM is alive.


---
## Step 1 — Check you actually got a GPU

If this says *NO GPU*, fix the runtime type and re-run. Nothing else will work without it.


In [ ]:
import shutil, subprocess
if not shutil.which('nvidia-smi'):
    print('NO GPU — Runtime > Change runtime type > L4 GPU > Save, then re-run this cell')
else:
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip())
    print('Good. Continue.')


---
## Step 2 — Install underfit

Downloads ~5 GB of Python packages (torch, CUDA libraries). Takes a couple of minutes.

**Success looks like:** a long list of packages ending in `underfit==0.1.0`.


In [ ]:
import os, subprocess
if not os.path.isdir('/content/underfit'):
    subprocess.run(['git','clone','--depth','1','https://github.com/dada-bots/underfit',
                    '/content/underfit'], check=True)
if not os.path.isdir('/content/stable-audio-3'):
    subprocess.run(['git','clone','--depth','1','https://github.com/Stability-AI/stable-audio-3',
                    '/content/stable-audio-3'], check=True)
r = subprocess.run(['./install.sh','--no-setup'], cwd='/content/underfit',
                   capture_output=True, text=True)
print(r.stdout[-800:])
print('EXIT', r.returncode)


---
## Step 3 — Mount Google Drive

Your Drive appears at `/content/drive/MyDrive`. This is where your dataset lives and where
you should save finished LoRAs — **everything else on this machine is deleted when the
session ends.**

A popup will ask you to authorise. Use **w2@2w12.one**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
print('Colab Notebooks folder contents:')
for f in sorted(os.listdir('/content/drive/MyDrive/Colab Notebooks'))[:20]:
    print('  ', f)


---
## Step 4 — HuggingFace token

The model weights live on HuggingFace. Two things are needed:

1. **One-time in a browser:** click *Agree and access repository* at
   <https://huggingface.co/stabilityai/stable-audio-3-medium> — this unlocks the ARC
   (demo) models. A token alone is **not** enough; without the click you get a 403.
2. **A read token:** <https://huggingface.co/settings/tokens>

Running this cell pops up a box. Paste the token there — it is not saved into the notebook.


In [ ]:
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('Paste your HuggingFace token: ')
print('token set, length', len(os.environ['HF_TOKEN']))


---
## Step 5 — Download the model (~24 GB)

Two checkpoints:

- **base** (14 GB) — the one that gets fine-tuned
- **ARC** (10 GB) — used to render demo audio during training

Takes several minutes. The cell prints progress as it goes; wait for `SETUP FINISHED`.

*These are re-downloaded every new session. That's normal and faster than storing them on Drive.*


In [ ]:
import subprocess, os, time
subprocess.Popen('cd /content/underfit && nohup uv run python -m underfit.cli.setup '
                 '--backend sa3 --backend-path /content/stable-audio-3 --models sa3-medium '
                 '> /content/setup.log 2>&1 &', shell=True, env=dict(os.environ))

def sh(c):
    return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()

# NOTE: the [u] bracket stops grep matching its own command line.
while True:
    running = sh("ps -eo args | grep -c '[u]nderfit.cli.setup'") or '0'
    size    = sh('du -sh /root/.cache/huggingface 2>/dev/null | cut -f1') or '0'
    print(f'downloaded: {size}   (setup running: {running})', flush=True)
    if running.strip() == '0':
        break
    time.sleep(30)

print(sh('tail -5 /content/setup.log'))
print('SETUP FINISHED')


---
## Step 6 — Unpack your dataset

Copies the latents zip from Drive and unpacks it.

**What latents are:** your audio, pre-compressed into the form the model trains on (~128×
smaller than WAV). mira's captions are stored inside them, so this one file carries
everything.

**Success looks like:** `38 latent files`.


In [ ]:
import subprocess, glob, os
ZIP = '/content/drive/MyDrive/Colab Notebooks/dune-ost-latents-same-l.zip'
assert os.path.exists(ZIP), f'Not found: {ZIP}  — drag the zip into Colab Notebooks on Drive'
os.makedirs('/content/latents', exist_ok=True)
subprocess.run(['unzip','-q','-o',ZIP,'-d','/content/latents'], check=True)
n = len(glob.glob('/content/latents/**/*.npy', recursive=True))
print(f'{n} latent files')
print('path for the dashboard:  /content/latents/sa3-medium')


---
## Step 7 — Launch the dashboard 👀

**This is where you see everything** — loss curves, demo audio, checkpoints.

Run the cell, wait ~20 seconds, then click the printed link.

Leave this cell running. Stopping it kills the dashboard (training keeps going).


In [ ]:
import subprocess, os, time
env = dict(os.environ)
env.update({'UNDERFIT_STATE_DIR':'/content/underfit/state',
            'UNDERFIT_MODELS_DIR':'/content/underfit/state/models'})
subprocess.Popen('cd /content/underfit && nohup uv run python dashboard/server.py '
                 '> /content/dashboard.log 2>&1 &', shell=True, env=env)
time.sleep(20)
from google.colab import output
print('Dashboard starting. Open it here:')
output.serve_kernel_port_as_window(8787)


---
## Step 8 — Settings to enter in the dashboard

Use these exact values. They were worked out by testing — see the notes under each.

### Dataset

| Field | Value |
|---|---|
| Path | `/content/latents/sa3-medium` |
| Model | **SA3-medium** |

### Finetune

| Field | Value |
|---|---|
| Base model | SA3-medium |
| Latent seq length | **2048** |
| Crop mode | Random |
| LoRA type | DoRA-rows |
| Rank | 16 |
| Alpha | = rank |
| LR | 1e-4 |
| Max steps | 10000 |
| Batch size | 1 |
| Ckpt every | 1000 |
| Demo every | 1000 |

> **Latent seq length.** Medium's native 4096 = 380 s, but most Dune cues are shorter, so
> ~29% of every step would be padding. 2048 (190 s) fits 27 of 38 tracks fully.

### Dataset Text Prompts — the important screen

| Setting | Value |
|---|---|
| Use fixed prompt | ❌ **OFF** |
| Prepend to prompt | ✅ ON — `zvq`, 80% |
| Use tags | ✅ ON |
| Tag pills ON | `TrackType`, `VocalType`, `genre`, `instruments`, `moods`, `bpm`, `keyscale` |
| Tag pills OFF | `prompt`, `trigger`, `length_seconds` |
| shuffle | ✅ ON |
| Balance bar | must read **Tags 100%** |

> **Why fixed is off.** It's a *competing* prompt source, not a modifier. Left on at 50%,
> half your steps train on the literal text `Genre: dune-ost` instead of real captions.
>
> **Why `prompt` and `trigger` are off.** `prompt` already contains every other field —
> leaving it on trains each caption twice. `trigger` would render as `trigger: zvq`
> instead of a clean prepended token.
>
> **Why `length_seconds` is off.** 8 tracks were cut at the 600 s encode cap, so their
> caption length disagrees with the actual audio. Duration is conditioned numerically anyway.

### Demos

- Demo 0: `zvq, TrackType: Music, VocalType: Instrumental, Genre: Electronic: Ambient, Moods: dark, epic, film, Instruments: synthesizer, strings, low brass`
- Demo 1: leave empty *(unconditional — your control: shows the base model isn't swallowed)*
- Steps: **8** on both. CFG 1. Different seeds.


---
## Step 9 — Save your LoRA to Drive

**Do this before ending the session** or the checkpoints are gone.

Run it any time — it copies whatever checkpoints exist so far.


In [ ]:
import glob, shutil, os
dest = '/content/drive/MyDrive/Colab Notebooks/sa3-loras'
os.makedirs(dest, exist_ok=True)
found = glob.glob('/content/underfit/state/runs/**/*.safetensors', recursive=True)
for f in found:
    shutil.copy2(f, os.path.join(dest, os.path.basename(f)))
    print('saved', os.path.basename(f))
print(f'\n{len(found)} checkpoint(s) -> {dest}')


---
## Step 10 — Check on training any time

Run this cell whenever you want a status readout without leaving the notebook.


In [ ]:
import subprocess
print(subprocess.run('nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total '
                     '--format=csv,noheader', shell=True, capture_output=True, text=True).stdout)
print('--- latest training output ---')
print(subprocess.run('tail -n 3 $(ls -t /content/underfit/state/runs/*.log 2>/dev/null | head -1) '
                     '2>/dev/null || echo "no run log yet"',
                     shell=True, capture_output=True, text=True).stdout)
print('--- checkpoints so far ---')
print(subprocess.run('ls -1 /content/underfit/state/runs/**/checkpoints/*.safetensors 2>/dev/null | tail -5 || echo none', shell=True, capture_output=True, text=True).stdout)


---
## ⚠️ Finishing up

1. Run **Step 9** to save your LoRAs to Drive.
2. **Runtime → Disconnect and delete runtime.**

Compute units are consumed for as long as the VM is alive, whether or not it's training.

### Using the LoRA

Download the `.safetensors` from Drive and run it **on your Mac** — medium inference needs
only ~5 GB and runs comfortably on 16 GB:

```bash
cd sa3-studio/stable-audio-3/optimized/mlx
.venv/bin/python scripts/sa3_mlx.py --dit medium \
  --lora ~/Downloads/dune-zvq.safetensors --lora-strength 0.7 \
  --prompt 'zvq, TrackType: Music, Moods: dark, epic, Instruments: strings, low brass' \
  --seconds 120 --out dune-test.wav
```

Train in the cloud, generate at home.
